# Actividad 1 – Migración de Base de Datos

**Autor:** Jhon Jader Benitez Valderrama 
**Grupo:** 10 
**Empresa:** Portosoft S.A.S  
**Materia:** Bases de Datos Analíticas  

---

## Contexto

Portosoft es una empresa dedicada a ofrecer soluciones tecnológicas y servicios de desarrollo de software.  
Actualmente, la empresa desea **migrar la información de vacantes y empleos** que se encuentra almacenada en archivos CSV a una **base de datos relacional moderna en Databricks**.

El objetivo principal es organizar de manera estructurada la información sobre los empleos publicados, empresas, ubicaciones y modalidades laborales, permitiendo mantener la integridad de los datos tras la migración y habilitando futuras integraciones con otros sistemas de reclutamiento.

---

## Dataset seleccionado

**Nombre:** LinkedIn Software Engineer Jobs Dataset  
**Fuente:** [https://www.kaggle.com/datasets/andresionek/linkedinjobs-dataset](https://www.kaggle.com/datasets/andresionek/linkedinjobs-dataset)  
**Autor:** Andrés Ionel  

## Variables Relevantes

**job_title (Título del trabajo):** Indicates the name or position of the job offer, e.g., Software Engineer, Data Analyst.
→ Muestra el nombre o puesto del empleo ofrecido.

**company (Empresa):** Represents the organization or employer offering the job position.
→ Identifica la empresa que publica la oferta laboral.

**job_location (Ubicación del trabajo):** Specifies the city, state, or region where the job is based.
→ Indica el lugar geográfico donde se desarrolla el trabajo.

**job_link (Enlace del trabajo):** Contains the direct LinkedIn URL of the job posting.
→ Es el enlace directo a la oferta en LinkedIn.

**first_seen (Primera aparición):** Shows the date when the job posting was first detected or listed.
→ Indica la fecha en que la oferta fue publicada o registrada por primera vez.

**search_city (Ciudad de búsqueda):** Identifies the city used as a filter or parameter during the dataset collection.
→ Indica la ciudad desde la cual se realizó la búsqueda o filtrado del empleo.

**search_country (País de búsqueda):** Shows the country associated with the search or job listing.
→ Representa el país donde se publicó o se encontró la oferta.

**job_level (Nivel del trabajo):** Defines the seniority or experience level required for the job (e.g., Associate, Mid-senior).
→ Describe el nivel de experiencia requerido para el cargo.

**job_type (Tipo de trabajo):** Specifies whether the job is remote, onsite, or hybrid.
→ Indica la modalidad del trabajo (presencial, remoto o híbrido).

**job_summary (Resumen del trabajo):** Provides a brief textual description of the job’s responsibilities.
→ Resume las funciones principales del puesto ofrecido.

**job_skills (Habilidades del trabajo):** Lists the technical or soft skills required for the position.
→ Enumera las habilidades necesarias para desempeñar el trabajo (por ejemplo: Python, SQL, AWS).

**Motivo de selección:**  
Este conjunto de datos contiene información realista sobre empleos del sector tecnológico.  
Resulta ideal para simular la migración de una base de datos de ofertas laborales hacia un entorno relacional moderno, tal como Portosoft requiere para su sistema de gestión de talento.


## Creamos la base de datos y cargamos los datos

In [0]:
!pip install kagglehub[pandas-datasets]>=0.3.8

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# importamos la librerias nesesarias
import pandas as pd
import os
import kagglehub
import zipfile

## funciones para descargar, estraer y leer el Dataset desde Kaggle

In [0]:
def download_dataset_zip(url = ""):
        print("Descargando dataset desde Kaggle...")
        dataset_path = kagglehub.dataset_download(url)
        print("Ruta al dataset:", dataset_path)
        return dataset_path
    
def extract_zip_files(dataset_path):
        zip_files = [f for f in os.listdir(dataset_path) if f.endswith('.zip')]
        if zip_files:
            zip_file = os.path.join(dataset_path, zip_files[0])
            extract_dir = os.path.join(dataset_path, "extracted")
            os.makedirs(extract_dir, exist_ok=True)
            print(f"Extrayendo {zip_file} en {extract_dir}...")
            with zipfile.ZipFile(zip_file, "r") as z:
                z.extractall(extract_dir)
            return extract_dir
        else:
            # Si no se encuentra un ZIP, se verifica si existen archivos CSV en la ruta
            csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
            if csv_files:
                print("No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.")
                return dataset_path
            else:
                raise FileNotFoundError("No se encontró ningún archivo .zip ni archivos .csv en la ruta del dataset")

def create_csv(csv_dir, csv_name=None):
    if csv_name:
        file_path = os.path.join(csv_dir, csv_name)
        print(f"Leyendo {file_path}...")
        df = pd.read_csv(file_path, encoding="latin1")
        print("CSV creado correctamente")
        return df
    else:
        csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError("No se encontraron archivos CSV en el directorio extraído")
        for file in csv_files:
            file_path = os.path.join(csv_dir, file)
            print(f"Leyendo {file_path}...")
            df = pd.read_csv(file_path, encoding="latin1")
        print("CSV creado correctamente")
        return df

    

In [0]:
# descargamos el dataset
df = pd.DataFrame()
dataset_path = download_dataset_zip("asaniczka/software-engineer-job-postings-linkedin")
# extraemos el zip
extract_dir = extract_zip_files(dataset_path)
# creamos el csv
df = create_csv(extract_dir)

Descargando dataset desde Kaggle...
Ruta al dataset: /home/spark-43d75808-690e-4f9e-ab2d-66/.cache/kagglehub/datasets/asaniczka/software-engineer-job-postings-linkedin/versions/3
No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.
Leyendo /home/spark-43d75808-690e-4f9e-ab2d-66/.cache/kagglehub/datasets/asaniczka/software-engineer-job-postings-linkedin/versions/3/postings.csv...
CSV creado correctamente


In [0]:
# comprobamos que el dataset exista, mirando las cabeceras
df.head()

,job_title,company,job_location,job_link,first_seen,search_city,search_country,job level,job_type,job_summary,job_skills
0,C# Software Engineer,E Tech Group,"West Chester, OH",https://www.linkedin.com/jobs/view/c%23-softwa...,2023-12-25,Covington,United States,Associate,Remote,"At E Tech Group, joining our team means joinin...","C#, .NET, WPF, ASP.NET MVC, WebAPI, C++, Progr..."
1,Software Implementation Engineer,Kardex,"Cincinnati, OH",https://www.linkedin.com/jobs/view/software-im...,2023-12-25,Covington,United States,Associate,Remote,Kardex Remstar is looking for a\nSoftware Impl...,"Software Implementation, Software Testing, Sof..."
2,"Senior Software Engineer, Back End (Go, AWS, J...",Jobs for Humanity,"Chattanooga, TN",https://www.linkedin.com/jobs/view/senior-soft...,2023-12-25,Chattanooga,United States,Mid senior,Onsite,Company Description\nJobs for Humanity is part...,"Java, Python, SQL, Node.js, Go, Scala, AWS, GC..."
3,"Senior Manager, Software Engineering, Full Stack",Jobs for Humanity,"Chattanooga, TN",https://www.linkedin.com/jobs/view/senior-mana...,2023-12-25,Chattanooga,United States,Mid senior,Onsite,Company Description\nJobs for Humanity is part...,"JavaScript, Java, TypeScript, SQL, Python, Go,..."
4,"Lead Software Engineer, Full Stack(JavaScript/...",Jobs for Humanity,"Chattanooga, TN",https://www.linkedin.com/jobs/view/lead-softwa...,2023-12-25,Chattanooga,United States,Mid senior,Onsite,Company Description\nJobs for Humanity is part...,"JavaScript, Java, AWS, Vue.js, HTML/CSS, TypeS..."


In [0]:
# convertimos el dataframe  de pandas a spark
spark_df = spark.createDataFrame(df)

## creacion de la tabla e insercion de datos

In [0]:
# remonbro una columna con espacio
spark_df = spark_df.withColumnRenamed("job level", "job_level")


In [0]:
#creamos la tabla
spark_df.write.mode("overwrite").saveAsTable("portosoft_db")

# consultas de sql

In [0]:
#verificamos la creacion correcta de la tabla 
spark.sql("SELECT * FROM portosoft_db LIMIT 5").show()


+--------------------+--------------------+--------------------+--------------------+----------+---------------+--------------+----------+--------+--------------------+--------------------+
|           job_title|             company|        job_location|            job_link|first_seen|    search_city|search_country| job_level|job_type|         job_summary|          job_skills|
+--------------------+--------------------+--------------------+--------------------+----------+---------------+--------------+----------+--------+--------------------+--------------------+
|   Software Engineer|Veridian Tech Sol...|        Johnston, IA|https://www.linke...|2023-12-25|West Des Moines| United States|Mid senior|  Onsite|Job type: Permane...|Typescript, JavaS...|
|   Software Engineer|       Source Allies|Des Moines Metrop...|https://www.linke...|2023-12-25|West Des Moines| United States|Mid senior|  Onsite|Source Allies is ...|Software Developm...|
|Senior Software E...|         Accroid Inc|      D

1. conteo de registros

In [0]:

%sql
SELECT COUNT(*) FROM portosoft_db;

COUNT(*)
9380


2. Nombres y Tipos de Columnas

In [0]:
%sql
DESCRIBE TABLE portosoft_db;

col_name,data_type,comment
job_title,string,null
company,string,null
job_location,string,null
job_link,string,null
first_seen,string,null
search_city,string,null
search_country,string,null
job_level,string,null
job_type,string,null
job_summary,string,null


la consulta realizada, está diciéndo qué columnas tiene tu tabla portosoft_db, qué tipo de datos usa cada una y si tienen algún comentario asociado (en este caso, ninguno)

1. consulta con filtro

In [0]:

%sql
SELECT *
FROM portosoft_db
WHERE job_title = 'Software Engineer'
LIMIT 10;

job_title company job_location job_link first_seen search_city search_country job_level job_type job_summary job_skills Software Engineer VeeAR Projects Inc. San Francisco, CA https://www.linkedin.com/jobs/view/software-engineer-at-veear-projects-inc-3752333254 2023-12-25 Novato United States Mid senior Onsite Job Title: - Software Engineer
Job Location: - San Francisco, CA (On-site)
Employment type: - 12+ Months Contract
Job Description: -
Development of an NFC based solution for authenticating a user using a mobile device with NFC (iPhone & Android) and an embedded NFC board on a sliding door.
Deep NFC experience beyond credit card readers
experience in embedded automotive systems
Wireless antenna design expertise in automotive applications
Knowledge of NFC app development on both iOS and Android system
Skills:
Developing NFC application for ios and Android devices
Development of an NFC-based solution for authenticating a user.
Show more
Show less NFC, iPhone, Android, iOS, Automotive Systems, Wireless Antenna Design, NFC App Development Software Engineer Guidewheel San Francisco, CA https://www.linkedin.com/jobs/view/software-engineer-at-guidewheel-3732800426 2023-12-25 Novato United States Mid senior Remote The Company
Guidewheel is on a mission to empower all the world's factories to reach sustainable peak performance. Inspired by the simple, universal truth that every machine on the factory floor has a power cord, our plug-and-play FactoryOps platform makes the power of the cloud accessible to any factory. Guidewheel clips onto any machine to turn its real-time "heartbeat" into a connected, actively learning system that empowers teams to reduce lost production time, increase throughput, and perform better and better over time.
At Guidewheel we work with the factories that are the backbone of our economy, and you can have a real, on-the-ground impact right away. We have strong momentum and alignment around our mission, investor support, and a culture that values diversity, growth mindset, and results. And the tight link between our mission and our business model means that reaching more of the world's 10 million factories accelerates our positive impact on the planet.
The Guidewheel team shares the following values:
Integrity matters: We are honest, straightforward and sincere. With each other. With our investors. With our customers.
We (actually) care: About each other. About fighting climate change. About making a real impact.
We use data to make decisions: We possess the courage to accept "hard truths" and confront challenges head-on.
The power of growth mindset is real: We strive to be the best we can be; to achieve this we are committed to embracing change and expanding our capabilities.
Role And Responsibilities
Executes well, but doesn't just execute â asks "why," understands the big picture, and delivers the best possible solution.
Completes high quality work in a timely manner.
Works closely with Tech lead/s to come up with the best design.
Willingness to learn and taking up any challenge.
Passionate about solving customer pain points.
Thoughtfully translates customer and business needs into software solutions that can scale.
Writes code that is designed to be future proof - not just within a function, but within a project's architecture.
Writes useful tests and encourages others to write tests.
Maintains clear documentation and proactively communicates.
Offers thoughtful code reviews, and constantly seeks improvement in their own work.
WHAT'S IN IT FOR YOU?
Work with cutting-edge software technologies, including the latest machine learning and big data tools, on a constant stream (terabytes/week) of time-series machine data.
Develop core technology for a product-first startup.
Grow with a team that believes developing talent is important, and is building that as a core strength of the company from day one.
Work with a small founding team to determine the future of our technology and the tools we use to solve m

La consulta seleccionó todos los registros de la tabla portosoft_db donde el campo job_title es igual a “Software Engineer”, mostrando solo los primeros 10 resultados.
En otras palabras, filtró las ofertas de empleo relacionadas con ese cargo específico dentro del dataset de LinkedIn.